In [13]:
from minsearch import VectorSearch # search by Vector
from minsearch import Index        # search by index

In [14]:
from tqdm import tqdm

In [15]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [16]:
documents = [file.parse() for file in reader.read()]

In [34]:
len(documents)

72

In [17]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [35]:
len(chunks)

295

In [18]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [19]:
query = "How do I give the model access to tools?"

In [45]:
text_results = index.search(
        query=query, 
        num_results=5
    )

In [42]:
##texts = [doc["filename"] + " " + doc["content"] for doc in chunks]

In [22]:
from embedder import Embedder
embed = Embedder()

2026-09-03 01:32:27.877198097 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [36]:
import numpy as np

batch_size = 50
X = []

#for i in tqdm(range(0, len(chunks), batch_size)):
#    batch = chunks[i:i + batch_size]
#    print("-----")
#    print(batch)
#    batch_vectors = embed.encode_batch(batch)
#    X.extend(batch_vectors)
for chunk in chunks:
    batch_vectors = embed.encode(chunk['content'])
    X.append(batch_vectors)


X = np.array(X)

In [37]:
print(len(X))
print(len(chunks))

295
295


In [41]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [42]:
#q1 = "What metric do we use to evaluate a search engine?"
q1 = "How do I give the model access to tools?"
v1 = embed.encode(q1)

In [44]:
vector_results = vindex.search(v1, num_results=5)

In [47]:
for x in text_results:
    print(x['start'], x['filename'])

0 01-agentic-rag/lessons/14-agentic-loop.md
4000 01-agentic-rag/lessons/13-function-calling.md
5000 01-agentic-rag/lessons/13-function-calling.md
1000 01-agentic-rag/lessons/13-function-calling.md
3000 04-evaluation/lessons/02-ground-truth.md


In [48]:
for x in vector_results:
    print(x['start'], x['filename'])

2000 01-agentic-rag/lessons/01-intro.md
1000 04-evaluation/lessons/02-ground-truth.md
0 01-agentic-rag/lessons/16-other-frameworks.md
2000 01-agentic-rag/lessons/15-frameworks.md
4000 01-agentic-rag/lessons/13-function-calling.md


In [49]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [50]:
results = rrf([vector_results, text_results])

In [52]:
for x in results:
    print(x['start'], x['filename'])

4000 01-agentic-rag/lessons/13-function-calling.md
2000 01-agentic-rag/lessons/01-intro.md
0 01-agentic-rag/lessons/14-agentic-loop.md
1000 04-evaluation/lessons/02-ground-truth.md
0 01-agentic-rag/lessons/16-other-frameworks.md
